# M04-03 — Segmentación

[← Anterior](03-lab-kpis.ipynb) · [Siguiente →](../M05-analisis-avanzado/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

Clasificar clientes con venta cobrable en low / mid / high según GMV y contar cada banda.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M04-03-segmentacion.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — GMV por cliente

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

La segmentación es una agregación DESPUÉS de fijar el grano. Parto de sales cobrable.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** ~**211** clientes con al menos un paid.

**Por qué este paso.** Si agregas *todos* los clientes con left, inflas con GMV nulo.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, sum as fsum, countDistinct, when, lit

spark = get_spark("novashop-m04")
fact = spark.read.parquet(str(STAGING / "fact_lines"))
customers = spark.read.parquet(str(STAGING / "customers_clean"))
sales = fact.join(customers, "customer_id", "inner").where(col("is_billable"))
customer_gmv = sales.groupBy("customer_id", "country", "segment").agg(
    fsum("gmv_line").alias("gmv"),
    countDistinct("order_id").alias("orders"),
)
customer_gmv.orderBy(col("gmv").desc()).show(5)
print(customer_gmv.count())


### Paso 2 — Bandas de negocio

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Umbrales explícitos: <1000 low, <3000 mid, resto high. Encadena when bien (no solapes).

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `high` ≈ 40 · `low` ≈ 56 · `mid` ≈ 115. Suma = count de customer_gmv.

**Por qué este paso.** Los quintiles (`ntile`) van en la mejora, no aquí.


In [ ]:
banded = customer_gmv.withColumn(
    "value_band",
    when(col("gmv") < 1000, lit("low"))
    .when(col("gmv") < 3000, lit("mid"))
    .otherwise(lit("high")),
)
banded.groupBy("value_band").count().orderBy("value_band").show()


### Paso 3 — Guarda para M05

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

M05 rankea sobre este grano sin recalcular el GMV.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Carpeta `data/staging/customer_gmv` y el mismo count (~211).

**Por qué este paso.** overwrite para poder repetir el lab.


In [ ]:
banded.write.mode("overwrite").parquet(str(STAGING / "customer_gmv"))
print(spark.read.parquet(str(STAGING / "customer_gmv")).count())


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

`low + mid + high` debe igualar `customer_gmv.count()`. Una sola cifra, sin clientes en dos bandas.


## Mejora — Quintiles

Usa `ntile(5)` sobre `gmv` (ventana global `orderBy(gmv)`) y cuenta cada quintil. Sin partitionBy: ranking de la compañía.

Si te atasca, el código está en la celda siguiente.


In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import ntile

w = Window.orderBy(col("gmv"))
customer_gmv.withColumn("q", ntile(5).over(w)).groupBy("q").count().orderBy("q").show()


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| 250 clientes en las bandas | Left con GMV nulo | Parte de sales cobrable |
| Un cliente en two bands | Whens solapados | `< 1000` luego `< 3000` luego otherwise |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M05 — teoría](../M05-analisis-avanzado/01-teoria.ipynb).
